# Tutorial 5: Newton and Quasi-Newton Methods

**Prerequisites:** Tutorial 1 — What Is an Optimization Problem?, Tutorial 2 — Types of Optimization Problems, Tutorial 3 — Gradient Descent from Scratch, Tutorial 4 — Finite Differences and Numerical Gradients  
**Up Next:** Tutorial 6 — KKT Conditions

---

## Concept

Gradient descent uses only first-order (gradient) information. It treats every direction the same, which makes it slow on problems where the objective curves sharply in one direction but gently in another. **Newton's method** fixes this by also using second-order information — the **Hessian** matrix $H$ of second derivatives — to take steps that account for curvature.

The catch: computing and inverting the Hessian is expensive. **Quasi-Newton methods** (most famously **BFGS**) build an *approximation* to the inverse Hessian from successive gradient evaluations, getting most of Newton's speed at a fraction of the cost.

## Key Takeaway

> **BFGS approximates the Hessian from gradient history, giving Newton-like convergence without computing second derivatives. It typically converges in far fewer iterations than steepest descent.**

## Math Core

**Newton's method** solves a local quadratic model at each step:

$$\mathbf{x}_{k+1} = \mathbf{x}_k - H_k^{-1} \nabla f(\mathbf{x}_k)$$

where $H_k = \nabla^2 f(\mathbf{x}_k)$ is the Hessian. This converges quadratically near a minimum but requires $O(n^2)$ storage and $O(n^3)$ per-step cost.

**BFGS** replaces $H_k^{-1}$ with an approximation $B_k^{-1}$ updated via a rank-2 formula:

$$B_{k+1}^{-1} = \left(I - \rho_k s_k y_k^T\right) B_k^{-1} \left(I - \rho_k y_k s_k^T\right) + \rho_k s_k s_k^T$$

where $s_k = \mathbf{x}_{k+1} - \mathbf{x}_k$, $y_k = \nabla f_{k+1} - \nabla f_k$, and $\rho_k = 1 / (y_k^T s_k)$. No second derivatives needed — only gradients.

## Code

We reuse the four-bar objective from previous tutorials, then compare gradient descent against `scipy.optimize.minimize` with `method='L-BFGS-B'`.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from dms.curves import CompareCurves
from scipy.optimize import minimize
%matplotlib inline

# Constants (same as Tutorial 1)
L0, L1 = 1.0, 1.0
PX, PY = 0.5, 0.3
TARGETS = np.array([
    [1.0, 1.5], [0.75, 1.5], [0.5, 1.5],
    [0.25, 1.5], [0.0, 1.5], [-0.25, 1.5]
])
THETAS = np.linspace(np.deg2rad(60), np.deg2rad(120), len(TARGETS))

In [ ]:
def fourbar_fk(theta1, l0, l1, l2, l3):
    """Cosine-law closed-form FK for four-bar linkage."""
    ax, ay = l1 * np.cos(theta1), l1 * np.sin(theta1)
    dx, dy = ax - l0, ay
    d = np.sqrt(dx**2 + dy**2)
    cos_alpha = (l3**2 + d**2 - l2**2) / (2 * l3 * d)
    if np.abs(cos_alpha) > 1:
        return None
    alpha = np.arccos(np.clip(cos_alpha, -1, 1))
    beta = np.arctan2(dy, dx)
    theta3 = beta + alpha
    bx = l0 + l3 * np.cos(theta3)
    by = l3 * np.sin(theta3)
    theta2 = np.arctan2(by - ay, bx - ax)
    return theta2, theta3


def coupler_point(theta1, l2, l3):
    """Compute coupler point for given crank angle."""
    sol = fourbar_fk(theta1, L0, L1, l2, l3)
    if sol is None:
        return None
    theta2, _ = sol
    ax, ay = L1 * np.cos(theta1), L1 * np.sin(theta1)
    ux, uy = np.cos(theta2), np.sin(theta2)
    return np.array([ax + PX * ux - PY * uy,
                     ay + PX * uy + PY * ux])


def objective(x):
    """Path-synthesis objective."""
    l2, l3 = x
    path = []
    for theta1 in THETAS:
        pt = coupler_point(theta1, l2, l3)
        if pt is None:
            return 1e3
        path.append(pt)
    path = np.array(path)
    return CompareCurves(path[:, 0], path[:, 1],
                         TARGETS[:, 0], TARGETS[:, 1])

### Gradient descent baseline

First, let's run the steepest-descent method from Tutorial 3 with a backtracking line search, so we have a fair baseline to compare against BFGS.

In [ ]:
def numerical_grad(f, x, eps=1e-7):
    """Central-difference gradient."""
    x = np.asarray(x, dtype=float)
    g = np.zeros_like(x)
    for i in range(len(x)):
        xp, xm = x.copy(), x.copy()
        xp[i] += eps;  xm[i] -= eps
        g[i] = (f(xp) - f(xm)) / (2 * eps)
    return g


def gradient_descent(f, x0, n_iter=80):
    """Steepest descent with backtracking line search."""
    x = np.array(x0, dtype=float)
    path = [x.copy()]
    for _ in range(n_iter):
        g = numerical_grad(f, x)
        if np.linalg.norm(g) < 1e-6:
            break
        alpha = 0.1
        f0 = f(x)
        for _ in range(20):
            xn = x - alpha * g
            if xn.min() > 0.3 and f(xn) < f0 - 1e-8:
                break
            alpha *= 0.5
        else:
            break
        x = xn
        path.append(x.copy())
    return np.array(path)

In [ ]:
x0 = np.array([1.0, 1.5])
gd_path = gradient_descent(objective, x0, n_iter=80)
gd_costs = [objective(p) for p in gd_path]
print(f'GD:  {len(gd_path)-1} iterations, '
      f'f = {gd_costs[0]:.4f} -> {gd_costs[-1]:.4f}')

### BFGS via scipy

Now let's run `L-BFGS-B` — a bounded variant of BFGS that naturally avoids stepping into regions where the mechanism cannot assemble.

In [ ]:
bfgs_path = [x0.copy()]

def bfgs_callback(xk):
    bfgs_path.append(xk.copy())

res_bfgs = minimize(objective, x0, method='L-BFGS-B',
                    callback=bfgs_callback,
                    bounds=[(0.3, 3.0), (0.3, 3.0)],
                    options={'ftol': 1e-12, 'eps': 1e-7})
bfgs_path = np.array(bfgs_path)
bfgs_costs = [objective(p) for p in bfgs_path]

print(f'BFGS: {len(bfgs_path)-1} iterations, '
      f'f = {bfgs_costs[0]:.4f} -> {bfgs_costs[-1]:.4f}')
print(f'Solution: l2={res_bfgs.x[0]:.4f}, l3={res_bfgs.x[1]:.4f}')

### Convergence comparison

Let's compare how quickly each method reduces the objective.

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(range(len(gd_costs)), gd_costs, 'b-o', ms=4, lw=2,
        label=f'Steepest descent ({len(gd_costs)-1} iters)')
ax.plot(range(len(bfgs_costs)), bfgs_costs, 'r-s', ms=6, lw=2,
        label=f'L-BFGS-B ({len(bfgs_costs)-1} iters)')
ax.set_xlabel('Iteration')
ax.set_ylabel(r'$f(\mathbf{x})$')
ax.set_title('Convergence: steepest descent vs. L-BFGS-B')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()

### Trajectories on the objective landscape

Overlaying both paths on the contour plot shows how BFGS uses curvature information to take more direct steps.

In [ ]:
# Build contour grid
l2v = np.linspace(0.3, 2.5, 80)
l3v = np.linspace(0.3, 2.5, 80)
L2, L3 = np.meshgrid(l2v, l3v)
Z = np.vectorize(lambda a, b: objective([a, b]))(L2, L3)

fig, ax = plt.subplots(figsize=(7, 5))
cf = ax.contourf(L2, L3, np.log1p(Z), levels=30, cmap='viridis')
ax.plot(gd_path[:, 0], gd_path[:, 1], 'b.-', ms=4, lw=1.5,
        label='Steepest descent')
ax.plot(bfgs_path[:, 0], bfgs_path[:, 1], 'r.-', ms=6, lw=2,
        label='L-BFGS-B')
ax.plot(*x0, 'wo', ms=8, zorder=5, label='start')
ax.set_xlabel(r'$\ell_2$')
ax.set_ylabel(r'$\ell_3$')
ax.set_title('Optimization trajectories')
ax.legend()
plt.colorbar(cf, label=r'$\ln(1 + f)$')
plt.tight_layout()

### How BFGS builds curvature information

BFGS starts with the identity matrix as its inverse-Hessian estimate (equivalent to steepest descent). After each step, the rank-2 update incorporates what it learned about curvature from the gradient change. After just a few iterations, the approximation captures the local shape of the objective well enough to take near-optimal steps.

This is why BFGS often converges in far fewer iterations than gradient descent — it adapts its step direction *and* step length to the geometry of the problem, without ever computing a second derivative.

## Mechanism Hook

Let's visualize the mechanism at the BFGS solution.

In [ ]:
from dms.mechanisms.fourbar import FourBar

l2, l3 = res_bfgs.x
mechanism = FourBar(L0, L1, l2, l3)
path = np.array([coupler_point(t, l2, l3) for t in THETAS])

fig, ax = plt.subplots(figsize=(6, 5))
mechanism.plot(np.deg2rad(90), ax=ax)
ax.plot(TARGETS[:, 0], TARGETS[:, 1], 'r*', ms=10, label='targets')
ax.plot(path[:, 0], path[:, 1], 'b.-', label='coupler path')
ax.set_aspect('equal')
ax.legend()
ax.set_title(f'BFGS result: $\\ell_2$={l2:.3f}, $\\ell_3$={l3:.3f}'
             f'  |  f = {objective(res_bfgs.x):.4f}')
plt.tight_layout()

BFGS found a good solution efficiently. But this was *unconstrained* — the algorithm was free to choose any link lengths. In Tutorial 6, we introduce **constraints** and the KKT conditions that characterize constrained optima.

---